In [ ]:
!git clone https://github.com/maduekedickson/natlas-challenge.git
%cd natlas-challenge


In [ ]:
!pip install -q -r requirements.txt peft fastapi uvicorn pydantic nest_asyncio streamlit


In [ ]:
import os, getpass
from huggingface_hub import whoami

hf_token = getpass.getpass('Paste your HF token: ')
os.environ['HF_TOKEN'] = hf_token
print('Token valid, belongs to:', whoami(token=hf_token).get('name'))


In [ ]:
!pip uninstall -y torchao
!python training/train.py --epochs 3


In [ ]:
import subprocess, os, time, requests

os.environ['ADAPTER_PATH'] = 'training/output/lora-adapter/final'

api_log = open('api.log', 'w')
subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)

streamlit_log = open('streamlit.log', 'w')
subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app/app.py',
     '--server.port', '8501', '--server.address', '0.0.0.0',
     '--server.headless', 'true', '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=streamlit_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)

print('Waiting ~75s...')
time.sleep(75)
print(requests.get('http://localhost:8000/api/v1/health', timeout=5).json())

from google.colab.output import eval_js
print(eval_js('google.colab.kernel.proxyPort(8501)'))


# For N-ATLaS healthReport Extraction API 

In [ ]:
from google.colab.output import eval_js
print(eval_js('google.colab.kernel.proxyPort(8000)'))


### That prints a URL. Open it in your browser with /docs appended at the end — e.g

In [ ]:
https://xxxxx-8000.colab.googleusercontent.com/docs


# If you want to test it from inside a Colab cell instead of the browser:

In [ ]:
import requests
print(requests.post('http://localhost:8000/api/v1/analyze', json={
    'report_text': "Good morning, this is Ngozi from Owerri Primary Health Centre. We have 20 doses of measles vaccine, we are out of ORS, and we offer antenatal care. The facility is open today."
}).json())
